# 🏛️ DỰ ÁN VILAW-LLM: HUẤN LUYỆN SFT VỚI UNSLOTH & QLoRA
### Base Model: `Qwen/Qwen2.5-7B-Instruct` | Hardware: Google Colab T4 GPU (Free)

> **Pipeline gồm 6 bước:**
> 1. Cài đặt thư viện Unsloth & TRL tối ưu riêng cho GPU T4.
> 2. Nạp Base Model 4-bit (QLoRA) tiết kiệm 70% VRAM.
> 3. Gắn LoRA Adapter (r=16, alpha=32) vào Attention & FFN layers.
> 4. Định dạng dữ liệu Train/Validation theo chuẩn ChatML template.
> 5. Huấn luyện với SFTTrainer + Cơ chế DỪNG SỚM (Early Stopping & max_steps).
> 6. Kiểm thử suy luận (Inference Test) & Lưu LoRA Adapter.

## 1. Kiểm tra GPU & Cài đặt Unsloth

In [ ]:
# Kiểm tra GPU Tesla T4 16GB
!nvidia-smi

In [ ]:
# Cài đặt Unsloth và TRL bản tương thích tối ưu cho Colab
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.29" trl peft accelerate bitsandbytes
!pip install pyarrow pandas datasets

## 2. Nạp Base Model với Unsloth 4-bit (QLoRA)

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None        # Tự động nhận diện (Float16 cho T4)
load_in_4bit = True # Bật 4-bit quantization

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen2.5-7B-Instruct",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

## 3. Cấu hình LoRA Adapter (PEFT)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

## 4. Chuẩn bị Dữ liệu Huấn luyện & Đánh giá (ChatML Template)

In [ ]:
import os
import pandas as pd
from datasets import Dataset
from unsloth.chat_templates import get_chat_template

# 1. Gắn ChatML template chuẩn của Qwen2.5 vào tokenizer
tokenizer = get_chat_template(tokenizer, chat_template="chatml")

SYSTEM_PROMPT = (
    "Bạn là một chuyên gia tư vấn pháp luật Việt Nam am hiểu sâu sắc các quy định pháp luật. "
    "Hãy trả lời câu hỏi dựa trên các văn bản quy phạm pháp luật hiện hành, "
    "viện dẫn chính xác số Điều, Khoản, tên luật và đưa ra lập luận logic, rõ ràng."
)

def convert_to_dataset(df_subset):
    formatted = []
    for _, row in df_subset.iterrows():
        formatted.append({
            "conversations": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": row["question"]},
                {"role": "assistant", "content": row["answer"]}
            ]
        })
    ds = Dataset.from_pandas(pd.DataFrame(formatted))
    def apply_template(examples):
        texts = [
            tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False)
            for convo in examples["conversations"]
        ]
        return {"text": texts}
    return ds.map(apply_template, batched=True)

# 2. Đọc file Train & Val (Bạn upload 2 file này lên Colab qua icon Thư mục ở góc trái)
df_train = pd.read_parquet("legal_sft_train.parquet")
df_train_sample = df_train.sample(n=min(8000, len(df_train)), random_state=42)
train_dataset = convert_to_dataset(df_train_sample)
print(f"✓ Đã nạp Train Dataset: {len(train_dataset):,} mẫu")

# Nạp Val Dataset nếu có
val_dataset = None
if os.path.exists("legal_sft_val.parquet"):
    df_val = pd.read_parquet("legal_sft_val.parquet")
    df_val_sample = df_val.sample(n=min(500, len(df_val)), random_state=42)
    val_dataset = convert_to_dataset(df_val_sample)
    print(f"✓ Đã nạp Validation Dataset: {len(val_dataset):,} mẫu")

print("\nVí dụ 1 mẫu format:")
print(train_dataset[0]["text"][:450])

## 5. Huấn luyện với SFTTrainer + Tự động Dừng Sớm (Early Stopping)

> 💡 **3 Cơ chế kiểm soát dừng train:**
> 1. **`max_steps = 250`:** Chỉ chạy đúng 250 steps (~15-20 phút), sau đó tự kết thúc.
> 2. **`EarlyStoppingCallback(early_stopping_patience=2)`:** Nếu sau 2 lần đo liên tiếp (mỗi 50 step) mà `eval_loss` không giảm thêm thì tự động ngắt và lấy checkpoint tốt nhất.
> 3. **Bấm nút Stop (■) thủ công:** Bất cứ lúc nào bạn thấy Loss đi ngang quanh 0.6 - 0.7, bấm Stop. Mô hình trong biến `model` vẫn giữ nguyên toàn bộ kiến thức đã học!

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments, EarlyStoppingCallback

training_args = TrainingArguments(
    output_dir="./vilaw-checkpoints",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,  # Effective Batch Size = 16
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    weight_decay=0.01,
    optim="adamw_8bit",
    fp16=True,
    logging_steps=10,
    # --- CẤU HÌNH DỪNG SỚM & GIỚI HẠN BƯỚC CHẠY ---
    max_steps=250,                  # Chạy tối đa 250 steps (~20 phút trên T4). Đặt -1 nếu muốn chạy hết epoch
    eval_strategy="steps" if val_dataset else "no",
    eval_steps=50,                  # Cứ 50 steps đánh giá trên tập Val 1 lần
    save_strategy="steps" if val_dataset else "no",
    save_steps=50,
    save_total_limit=2,             # Chỉ giữ 2 checkpoint tốt nhất
    load_best_model_at_end=True if val_dataset else False,
    metric_for_best_model="eval_loss" if val_dataset else None,
    greater_is_better=False,
    report_to="none",
    seed=42,
)

callbacks = [EarlyStoppingCallback(early_stopping_patience=2)] if val_dataset else []

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    packing=False,
    callbacks=callbacks,
    args=training_args,
)

print("🚀 Bắt đầu quá trình huấn luyện SFT...")
trainer_stats = trainer.train()

## 6. Kiểm thử Suy luận (Inference Test) Trực tiếp

In [ ]:
# Chuyển sang chế độ Inference tối ưu của Unsloth
FastLanguageModel.for_inference(model)

cau_hoi_test = "Thời hiệu khởi kiện vụ án tranh chấp hợp đồng thương mại được quy định như thế nào trong Luật Thương mại Việt Nam?"

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": cau_hoi_test}
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to("cuda")

outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=512,
    temperature=0.3,
    repetition_penalty=1.1
)

tra_loi = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
print("=== KẾT QUẢ VILAW-LLM TRẢ LỜI ===\n")
print(tra_loi)

## 7. Lưu LoRA Adapter

In [ ]:
model.save_pretrained("vilaw-sft-lora")
tokenizer.save_pretrained("vilaw-sft-lora")
print("✓ Đã lưu LoRA Adapter tại thư mục: vilaw-sft-lora (~160MB)")
print("Bạn có thể nén thư mục này (zip) để tải về máy hoặc lưu lên Google Drive!")

In [ ]:
import shutil
shutil.make_archive("vilaw-sft-lora", 'zip', "vilaw-sft-lora")
from google.colab import files
files.download("vilaw-sft-lora.zip")